# Машинное обучение, ФКН ВШЭ

## Практическое задание 11. Метод опорных векторов и аппроксимация ядер

### Общая информация

Дата выдачи: 06.05.2026

Мягкий дедлайн: 23:59MSK 18.05.2026

Жесткий дедлайн: 23:59MSK 23.05.2026

### Оценивание и штрафы
Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи). Максимальная оценка за работу (без учёта бонусов) — 10 баллов. Всего за работу с бонусами можно набрать 12.75 баллов.

Сдавать задание после указанного жёсткого срока сдачи нельзя.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов (подробнее о плагиате см. на странице курса). Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник в отдельном блоке в конце вашей работы (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник). 

Использование генеративных языковых моделей разрешено только в случае явного указания на это. Необходимо прописать (в соответствующих пунктах, где использовались, либо в начале/конце работы):
- какая языковая модель использовалась
- какие использовались промпты и в каких частях работы
- с какими сложностями вы столкнулись при использовании генеративных моделей, с чем они помогли больше всего

Неэффективная реализация кода может негативно отразиться на оценке.

### Формат сдачи
Задания сдаются через систему anytask. Посылка должна содержать:
* Ноутбук homework-practice-11-random-features-Username.ipynb

Username — ваша фамилия и имя на латинице именно в таком порядке

### О задании

На занятиях мы подробно обсуждали метод опорных векторов (SVM). В базовой версии в нём нет чего-то особенного — мы всего лишь используем специальную функцию потерь, которая не требует устремлять отступы к бесконечности; ей достаточно, чтобы отступы были не меньше +1. Затем мы узнали, что SVM можно переписать в двойственном виде, который, позволяет заменить скалярные произведения объектов на ядра. Это будет соответствовать построению модели в новом пространстве более высокой размерности, координаты которого представляют собой нелинейные модификации исходных признаков.

Ядровой SVM, к сожалению, довольно затратен по памяти (нужно хранить матрицу Грама размера $d \times d$) и по времени (нужно решать задачу условной оптимизации с квадратичной функцией, а это не очень быстро). Мы обсуждали, что есть способы посчитать новые признаки $\tilde \varphi(x)$ на основе исходных так, что скалярные произведения этих новых $\langle \tilde \varphi(x), \tilde \varphi(z) \rangle$ приближают ядро $K(x, z)$.

Мы будем исследовать аппроксимации методом Random Fourier Features (RFF, также в литературе встречается название Random Kitchen Sinks) для гауссовых ядер. Будем использовать формулы, которые немного отличаются от того, что было на лекциях (мы добавим сдвиги внутрь тригонометрических функций и будем использовать только косинусы, потому что с нужным сдвигом косинус превратится в синус):
$$\tilde \varphi(x) = (
\cos (w_1^T x + b_1),
\dots,
\cos (w_n^T x + b_n)
),$$
где $w_j \sim \mathcal{N}(0, 1/\sigma^2)$, $b_j \sim U[-\pi, \pi]$.

На новых признаках $\tilde \varphi(x)$ мы будем строить любую линейную модель.

Можно считать, что это некоторая новая парадигма построения сложных моделей. Можно направленно искать сложные нелинейные закономерности в данных с помощью градиентного бустинга или нейронных сетей, а можно просто нагенерировать большое количество случайных нелинейных признаков и надеяться, что быстрая и простая модель (то есть линейная) сможет показать на них хорошее качество. В этом задании мы изучим, насколько работоспособна такая идея.

### Алгоритм

Вам потребуется реализовать следующий алгоритм:
1. Понизить размерность выборки до new_dim с помощью метода главных компонент.
2. Для полученной выборки оценить гиперпараметр $\sigma^2$ с помощью эвристики (рекомендуем считать медиану не по всем парам объектов, а по случайному подмножеству из где-то миллиона пар объектов): $$\sigma^2 = \text{median}_{i, j = 1, \dots, \ell, i \neq j} \left\{\sum_{k = 1}^{d} (x_{ik} - x_{jk})^2 \right\}$$
3. Сгенерировать n_features наборов весов $w_j$ и сдвигов $b_j$.
4. Сформировать n_features новых признаков по формулам, приведённым выше.
5. Обучить линейную модель (логистическую регрессию или SVM) на новых признаках.
6. Повторить преобразования (PCA, формирование новых признаков) к тестовой выборке и применить модель.

In [ ]:
%load_ext autoreload
%autoreload 2


Тестировать алгоритм мы будем на данных Fashion MNIST. Ниже код для их загрузки и подготовки.

In [ ]:
import numpy as np

# 1 Способ
import keras
from keras.datasets import fashion_mnist
(x_train_pics, y_train), (x_test_pics, y_test) = fashion_mnist.load_data()

# 2 Способ (если первый не работает)
# from sklearn.datasets import fetch_openml
# def load_fashion_mnist():
#     X, y = fetch_openml('Fashion-MNIST', version=1, return_X_y=True, as_frame=False)
#     X = X.reshape(-1, 28, 28).astype('uint8')
#     y = y.astype('int64')
#     x_train, x_test = X[:60000], X[60000:]
#     y_train, y_test = y[:60000], y[60000:]
#     return (x_train, y_train), (x_test, y_test)
# (x_train_pics, y_train), (x_test_pics, y_test) = load_fashion_mnist()

x_train = x_train_pics.reshape(y_train.shape[0], -1).astype(np.float32) / 255.0
x_test = x_test_pics.reshape(y_test.shape[0], -1).astype(np.float32) / 255.0



__Задание 0. (0.25 баллов)__

**Вопрос:** зачем в алгоритме нужен метод главных компонент?

**Ответ:** PCA нужен по трем причинам. Во-первых, он уменьшает размерность исходных изображений Fashion MNIST с 784 признаков до `new_dim`, поэтому генерация RFF и последующее обучение становятся быстрее. Во-вторых, после PCA признаки становятся некоррелированными и упорядоченными по объясненной дисперсии, из-за чего медианная эвристика для ширины RBF-ядра обычно работает стабильнее. В-третьих, PCA частично убирает шумовые направления, которые почти не помогают классификации, но влияют на случайные проекции и расстояния между объектами.


__Задание 1. (3 балла)__

Реализуйте алгоритм, описанный выше. Можете воспользоваться шаблоном класса в `homework_practice_08_rff.py` (допишите его и исправьте несостыковки в классе пайплайна) или написать свой интерфейс.

Ваша реализация должна поддерживать следующие опции:
1. Возможность задавать значения гиперпараметров new_dim (по умолчанию 50) и n_features (по умолчанию 1000).
2. Возможность включать или выключать предварительное понижение размерности с помощью метода главных компонент.
3. Возможность выбирать тип линейной модели (логистическая регрессия или SVM с линейным ядром).

Протестируйте на данных Fashion MNIST, сформированных кодом выше. Если на тесте у вас получилась доля верных ответов не ниже 0.84 с гиперпараметрами по умолчанию, то вы всё сделали правильно.

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

from homework_practice_11_rff import RFFPipeline, RandomFeatureCreator

pipeline = RFFPipeline(
    n_features=1000,
    new_dim=50,
    use_PCA=True,
    feature_creator_class=RandomFeatureCreator,
    classifier_class=LogisticRegression,
    classifier_params={"C": 10.0, "max_iter": 1000, "n_jobs": -1, "multi_class": "auto"},
    random_state=42,
)

pipeline.fit(x_train, y_train)
y_pred = pipeline.predict(x_test)
rff_accuracy = accuracy_score(y_test, y_pred)
rff_accuracy



__Задание 2. (2.5 балла)__

Сравните подход со случайными признаками с обучением SVM на исходных признаках. Попробуйте вариант с обычным (линейным) SVM и с ядровым SVM. Ядровой SVM может очень долго обучаться, поэтому можно делать любые разумные вещи для ускорения: брать подмножество объектов из обучающей выборки, например.

Сравните подход со случайными признаками с вариантом, в котором вы понижаете размерность с помощью PCA и обучите градиентный бустинг. Используйте одну из реализаций CatBoost/LightGBM/XGBoost. 

Сделайте выводы — насколько идея со случайными признаками работает? Сравните как с точки зрения качества, так и с точки зрения скорости обучения и применения.

In [ ]:
import time
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC, SVC

from homework_practice_11_rff import RFFPipeline, RandomFeatureCreator

rng = np.random.default_rng(42)
train_subset = rng.choice(len(x_train), size=15000, replace=False)
test_subset = rng.choice(len(x_test), size=5000, replace=False)
Xtr, ytr = x_train[train_subset], y_train[train_subset]
Xte, yte = x_test[test_subset], y_test[test_subset]

results = []

def benchmark(name, model, X_train=Xtr, y_train=ytr, X_test=Xte, y_test=yte):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - start
    start = time.perf_counter()
    pred = model.predict(X_test)
    predict_time = time.perf_counter() - start
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "fit_time_sec": fit_time,
        "predict_time_sec": predict_time,
    })
    return model

benchmark(
    "RFF + LogisticRegression",
    RFFPipeline(
        n_features=1000,
        new_dim=50,
        feature_creator_class=RandomFeatureCreator,
        classifier_class=LogisticRegression,
        classifier_params={"C": 10.0, "max_iter": 1000, "n_jobs": -1},
        random_state=42,
    ),
)

benchmark("LinearSVC on raw pixels", LinearSVC(C=1.0, max_iter=5000, random_state=42))

# Ядровой SVM существенно дороже, поэтому обучаем его на меньшей подвыборке.
small_train = rng.choice(len(x_train), size=5000, replace=False)
benchmark(
    "RBF SVC on raw pixels, 5000 train objects",
    SVC(C=3.0, kernel="rbf", gamma="scale", random_state=42),
    x_train[small_train],
    y_train[small_train],
    Xte,
    yte,
)

# По условию сравниваем PCA + градиентный бустинг. Основной вариант — XGBoost;
# если пакет не установлен в окружении, остается близкий sklearn fallback для запуска ноутбука.
try:
    from xgboost import XGBClassifier
    boosting = XGBClassifier(
        n_estimators=250,
        max_depth=5,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
    )
    boosting_name = "PCA + XGBoost"
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    boosting = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.08, random_state=42)
    boosting_name = "PCA + HistGradientBoosting fallback"

benchmark(boosting_name, Pipeline([("pca", PCA(n_components=50, random_state=42)), ("boosting", boosting)]))

comparison = pd.DataFrame(results).sort_values("accuracy", ascending=False)
comparison



__Задание 3. (1.75 балла)__

Проведите эксперименты:
1. Помогает ли предварительное понижение размерности с помощью PCA?
2. Как зависит итоговое качество от n_features? Выходит ли оно на плато при росте n_features?
3. Важно ли, какую модель обучать — логистическую регрессию или SVM?

Предварительный вывод по сравнению из задания 2: RFF обычно занимает промежуточное место между линейной моделью на исходных пикселях и полноценным RBF SVM. Линейный SVM быстрый, но хуже ловит нелинейность. RBF SVM может давать сильное качество на подвыборке, но плохо масштабируется по времени и памяти. PCA + бустинг часто конкурентоспособен по качеству, но обучение бустинга на большом числе объектов обычно дольше, чем обучение линейной модели поверх RFF.


In [ ]:
from sklearn.svm import LinearSVC

experiment_rows = []

def run_rff_experiment(name, **kwargs):
    model = RFFPipeline(random_state=42, **kwargs)
    start = time.perf_counter()
    model.fit(Xtr, ytr)
    fit_time = time.perf_counter() - start
    pred = model.predict(Xte)
    experiment_rows.append({
        "experiment": name,
        "accuracy": accuracy_score(yte, pred),
        "fit_time_sec": fit_time,
    })

for use_pca in [True, False]:
    run_rff_experiment(
        f"PCA={use_pca}",
        n_features=1000,
        new_dim=50,
        use_PCA=use_pca,
        feature_creator_class=RandomFeatureCreator,
        classifier_class=LogisticRegression,
        classifier_params={"C": 10.0, "max_iter": 1000, "n_jobs": -1},
    )

for n_features in [100, 300, 1000, 2000, 5000]:
    run_rff_experiment(
        f"n_features={n_features}",
        n_features=n_features,
        new_dim=50,
        use_PCA=True,
        feature_creator_class=RandomFeatureCreator,
        classifier_class=LogisticRegression,
        classifier_params={"C": 10.0, "max_iter": 1000, "n_jobs": -1},
    )

run_rff_experiment(
    "RFF + LinearSVC",
    n_features=1000,
    new_dim=50,
    use_PCA=True,
    feature_creator_class=RandomFeatureCreator,
    classifier_class=LinearSVC,
    classifier_params={"C": 1.0, "max_iter": 5000},
)

experiments = pd.DataFrame(experiment_rows)
experiments



__Бонус. (Максимум 1.75 балла)__

Как вы, должно быть, помните с курса МО-1, многие алгоритмы машинного обучения работают лучше, если признаки данных некоррелированы. Оказывается, что для RFF существует модификация, позволяющая получать ортогональные случайные признаки (Orthogonal Random Features, ORF). Об этом методе можно прочитать в [статье](https://proceedings.neurips.cc/paper/2016/file/53adaf494dc89ef7196d73636eb2451b-Paper.pdf). Реализуйте класс для вычисления ORF по аналогии с основным заданием. Обратите внимание, что ваш класс должен уметь работать со случаем n_features > new_dim (в статье есть замечание на этот счет), n_features=new_dim и n_features < new_dim также должны работать, убедитесь в этом. Проведите эксперименты, сравнивающие RFF и ORF, сделайте выводы. 


In [ ]:
from homework_practice_11_rff import OrthogonalRandomFeatureCreator

orf_rows = []
for creator_name, creator_class in [
    ("RFF", RandomFeatureCreator),
    ("ORF", OrthogonalRandomFeatureCreator),
]:
    for n_features in [25, 50, 100, 500, 1000, 2000]:
        model = RFFPipeline(
            n_features=n_features,
            new_dim=50,
            use_PCA=True,
            feature_creator_class=creator_class,
            classifier_class=LogisticRegression,
            classifier_params={"C": 10.0, "max_iter": 1000, "n_jobs": -1},
            random_state=42,
        )
        start = time.perf_counter()
        model.fit(Xtr, ytr)
        fit_time = time.perf_counter() - start
        pred = model.predict(Xte)
        orf_rows.append({
            "features": creator_name,
            "n_features": n_features,
            "accuracy": accuracy_score(yte, pred),
            "fit_time_sec": fit_time,
        })

orf_comparison = pd.DataFrame(orf_rows)
orf_comparison



__Бонус. (Максимум 1 балл)__

Существует большое количество работ, где идея RFF развивается, предлагаются её обобщения (которые, по сути, выливаются в другие преобразования признаков, не обязательно уже тригонометрические). Возьмите любую из таких работ, кратко опишите идею, имплементируйте её и сравните качество с ORF и RFF, которые вы запрограммировали выше.

___ссылка на работу:___ https://arxiv.org/pdf/1407.5599

___описание идеи:___ В качестве простой вариации используем Random Binning Features для Laplace-like ядер: каждая случайная сетка разбивает пространство на интервалы, а объект кодируется номером корзины. Совпадение корзин для двух объектов дает приближение ядровой похожести. Ниже реализована легкая версия через случайные сдвиги и ширины бинов; для классификации после такого преобразования снова обучается линейная модель.


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from scipy import sparse

class RandomBinningFeatureCreator(BaseEstimator, TransformerMixin):
    def __init__(self, n_features=1000, new_dim=50, random_state=42, scale=1.0):
        self.n_features = n_features
        self.new_dim = new_dim
        self.random_state = random_state
        self.scale = scale
        self.widths = None
        self.shifts = None
        self.projections = None

    def fit(self, X, y=None):
        rng = np.random.default_rng(self.random_state)
        X = np.asarray(X)
        self.new_dim = X.shape[1]
        self.widths = rng.exponential(scale=self.scale, size=self.n_features) + 1e-6
        self.shifts = rng.uniform(0, self.widths)
        self.projections = rng.integers(0, self.new_dim, size=self.n_features)
        return self

    def transform(self, X, y=None):
        X = np.asarray(X)
        bins = np.floor((X[:, self.projections] + self.shifts) / self.widths).astype(np.int64)
        # Хешируем пары (номер случайной сетки, номер бина) в фиксированное пространство.
        hashed = (bins * 1000003 + np.arange(self.n_features)) % self.n_features
        rows = np.repeat(np.arange(X.shape[0]), self.n_features)
        cols = hashed.ravel()
        data = np.ones(rows.shape[0], dtype=np.float32) / np.sqrt(self.n_features)
        return sparse.csr_matrix((data, (rows, cols)), shape=(X.shape[0], self.n_features))

rbf_model = Pipeline([
    ("pca", PCA(n_components=50, random_state=42)),
    ("rbf_features", RandomBinningFeatureCreator(n_features=2000, random_state=42, scale=0.25)),
    ("classifier", LogisticRegression(max_iter=1000, n_jobs=-1)),
])

start = time.perf_counter()
rbf_model.fit(Xtr, ytr)
rbf_time = time.perf_counter() - start
rbf_pred = rbf_model.predict(Xte)
random_binning_result = {
    "features": "Random Binning",
    "n_features": 2000,
    "accuracy": accuracy_score(yte, rbf_pred),
    "fit_time_sec": rbf_time,
}
random_binning_result



__Задание 4. (Максимум 2.5 балла)__

Реализуйте класс ядровой Ridge регрессии (Лекция 13, $\S 1.2$), для оптимизации используте градиентный спуск **[1 балл максимум]**, также добавьте возможность использовать аналитическую формулу **[1 балл максимум]**. Для градиентного спуска выпишите градиент ниже **[0.5 баллов максимум]**. 
Подумайте о том, как в формулах правильно учесть свободный коэффициент. 

Затем адаптируйте вашу реализацию RFF под задачу регрессии. Сравните вашу ядровую регрессию и RFF на синтетических данных.

Функция потерь:
$$
Q(w) = \frac{1}{2} ||\Phi \Phi^T w - y||^2 + \frac{\lambda}{2} w^T \Phi \Phi^T w \rightarrow \min_w,
$$
где $\Phi \Phi^T = K$, $K = (k(x_i, x_j))_{i, j = 1}^{\ell}$.

Предсказание:
$
y(x) = k(x)^T w,
$
где $k(x)$ — вектор функций ядра от пар объектов $(x, x_i)_{i=1}^{\ell}$.

___Выведите градиент:___
$$
\nabla Q(w) = K^T(Kw - y) + \lambda K w.
$$
Так как матрица ядра симметрична, $K^T = K$, поэтому
$$
\nabla Q(w) = K(Kw - y) + \lambda K w.
$$
Свободный коэффициент можно учитывать отдельной константной координатой в признаках или центрированием целевой переменной; в реализованном сравнении ниже используется синтетическая функция без отдельного bias в ядровой части, а RFF-регрессор использует Ridge, где свободный коэффициент поддерживается стандартной моделью.

Вы можете изменять представленный шаблон в файле `homework_practice_11_kernel_regression.py` по своему усмотрению.


In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

from homework_practice_11_kernel_regression import KernelRidgeRegression
from homework_practice_11_rff import RFFRegressor

rng = np.random.default_rng(42)
X = np.linspace(-3, 3, 500).reshape(-1, 1)
y = np.sin(2 * X[:, 0]) + 0.3 * np.cos(5 * X[:, 0]) + rng.normal(0, 0.1, size=X.shape[0])
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y, test_size=0.3, random_state=42
)

krr_closed = KernelRidgeRegression(regularization=0.1, kernel_scale=0.5)
krr_closed.fit_closed_form(X_train_reg, y_train_reg)

krr_gd = KernelRidgeRegression(
    lr=1e-4,
    regularization=0.1,
    tolerance=1e-8,
    max_iter=3000,
    kernel_scale=0.5,
)
krr_gd.fit(X_train_reg, y_train_reg)

rff_reg = RFFRegressor(
    n_features=1000,
    new_dim=1,
    use_PCA=False,
    regressor_params={"alpha": 0.1},
    random_state=42,
)
rff_reg.fit(X_train_reg, y_train_reg)

regression_comparison = pd.DataFrame([
    {
        "model": "Kernel Ridge, closed form",
        "mse": mean_squared_error(y_test_reg, krr_closed.predict(X_test_reg)),
    },
    {
        "model": "Kernel Ridge, gradient descent",
        "mse": mean_squared_error(y_test_reg, krr_gd.predict(X_test_reg)),
        "iterations": len(krr_gd.loss_history),
    },
    {
        "model": "RFF + Ridge regression",
        "mse": mean_squared_error(y_test_reg, rff_reg.predict(X_test_reg)),
    },
])
regression_comparison



### Итоговые выводы

1. PCA для RFF полезен практически: он ускоряет оценку ширины ядра и генерацию признаков, а качество обычно не падает, потому что большая часть полезной информации Fashion MNIST сохраняется в первых компонентах.
2. При росте `n_features` качество RFF растет не линейно: заметный прирост обычно есть от сотен к тысяче признаков, а дальше начинается плато, где время и память растут сильнее, чем accuracy.
3. Логистическая регрессия и линейный SVM на RFF дают близкий смысловой результат: важнее само нелинейное отображение. Различия зависят от регуляризации и числа итераций.
4. ORF уменьшает дисперсию случайного приближения и часто выигрывает у обычного RFF при малом и среднем числе случайных признаков. При большом `n_features` разница становится меньше.
5. Ядровой Ridge в аналитической форме удобен на небольших выборках, но требует хранения полной матрицы ядра. RFF + Ridge масштабируется лучше, потому что заменяет ядро явными признаками и обучает обычную линейную модель.

### Использование генеративной модели

При подготовке решения использовался OpenAI Codex / ChatGPT-5 в роли помощника по программированию. Промпт: посмотреть файлы задания, не менять исходный ноутбук, реализовать недостающие классы в `.py` файлах и создать новый решенный ноутбук. Модель помогла быстрее заполнить шаблоны RFF, ORF и Kernel Ridge Regression, а также структурировать код экспериментов. Основная сложность — окружение в этой папке не содержит `scikit-learn`, поэтому тяжелые эксперименты и численные результаты нужно выполнить в учебном окружении с установленными зависимостями.
